# Python 与 LangGraph 交互式学习笔记

这份 Notebook 是 `python_langgraph_notes.md` 的配套版本：Markdown 单元负责解释概念，Code 单元负责演示可运行示例。

使用建议：

- 先顺序阅读 Markdown，再运行后面的代码单元。
- 大部分示例不需要真实大模型 API Key。
- DeepSeek 相关单元会显式读取项目根目录 `.env`，支持 `DEEPSEEK_API_KEY`，也兼容 OpenAI-style 的 `OPENAI_API_KEY` / `OPENAI_BASE_URL` / `OPENAI_MODEL`。
- 当前 LangGraph v1 之后推荐使用 `from langchain.agents import create_agent`，不要再用已弃用的 `langgraph.prebuilt.create_react_agent`。

## 1. LangGraph 是什么

LangGraph 是 LangChain 生态中用于构建有状态 Agent 和复杂 LLM 工作流的框架。

简单理解：

```text
LangChain 更偏组件：模型、Prompt、工具、链、检索器。
LangGraph 更偏编排：状态、节点、边、循环、暂停、恢复、多代理。
```

当你的应用需要工具调用、多轮状态、人工审批、流式过程、多 Agent 协作、恢复和调试时，LangGraph 就比单纯的链式调用更合适。

In [4]:
# 安装依赖示例。Notebook 中一般不自动执行安装命令。
# %pip install -U langgraph langchain langchain-deepseek langchain-openai python-dotenv

import os
from pathlib import Path

from dotenv import load_dotenv

# Notebook 的工作目录可能是 docs/，也可能是项目根目录。
# 这里向上查找 .env，确保能读到 D:/PythonProject/LearnOne/.env。
def find_project_env(start: Path | None = None) -> Path | None:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        env_path = path / ".env"
        if env_path.exists():
            return env_path
    return None


env_path = find_project_env()
if env_path:
    load_dotenv(env_path, override=True)
    print(f"已加载环境变量：{env_path}")
else:
    print("没有找到 .env，真实模型调用单元会跳过。")

print("DEEPSEEK_API_KEY 已配置：", bool(os.getenv("DEEPSEEK_API_KEY")))
print("OPENAI_API_KEY 已配置：", bool(os.getenv("OPENAI_API_KEY")))
print("LangGraph 学习 Notebook 已加载。")

已加载环境变量：D:\PythonProject\LearnOne\.env
DEEPSEEK_API_KEY 已配置： True
OPENAI_API_KEY 已配置： True
LangGraph 学习 Notebook 已加载。


## 2. DeepSeek + create_agent 基础写法

`create_agent` 是当前推荐的预构建 Agent 入口。这个单元只演示真实 DeepSeek 的普通对话调用，不绑定工具。

原因：真实模型的 tool calling 受模型、网关和接口兼容性影响，可能出现 502 等服务端错误。Notebook 里的工具调用和 context 会在后面的离线假模型单元演示，保证学习过程稳定。

In [6]:
import os

from langchain.agents import create_agent


def build_deepseek_model():
    """优先使用 langchain-deepseek，兼容 OpenAI-style DeepSeek 配置。"""

    if os.getenv("DEEPSEEK_API_KEY"):
        from langchain_deepseek import ChatDeepSeek

        return ChatDeepSeek(
            model=os.getenv("DEEPSEEK_MODEL", "deepseek-chat"),
            api_key=os.getenv("DEEPSEEK_API_KEY"),
            temperature=float(os.getenv("DEEPSEEK_TEMPERATURE", "0")),
        )

    if os.getenv("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI

        return ChatOpenAI(
            model=os.getenv("OPENAI_MODEL", "deepseek-chat"),
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=os.getenv("OPENAI_BASE_URL", "https://api.deepseek.com/v1"),
            temperature=0,
        )

    return None


model = build_deepseek_model()

if model is None:
    print("没有找到 DeepSeek/OpenAI-compatible API Key，跳过真实模型调用。")
else:
    agent = create_agent(
        model=model,
        tools=[],
        system_prompt="你是一个简洁可靠的中文助手。",
    )

    response = agent.invoke(
        {"messages": [{"role": "user", "content": "用一句话解释 LangGraph 是什么。"}]}
    )
    print(response["messages"][-1].content)

LangGraph 是一个用于构建有状态、多步骤语言模型工作流的框架，支持图结构编排和循环逻辑。


## 3. Agent 输入输出

LangGraph / LangChain Agent 的常见输入是：

```python
{"messages": [{"role": "user", "content": "你好"}]}
```

输出通常也是一个字典，其中最常用的是 `messages`。如果配置了结构化输出，还会有 `structured_response`。

## 4. Context 上下文：把运行时信息安全传给工具

`context` 适合传用户 ID、租户、权限、语言偏好、请求来源等运行时信息。

重点：这些信息不是让模型自己编，而是由代码在 `agent.invoke(..., context=...)` 时传入，工具通过 `ToolRuntime[Context]` 读取。

下面这个例子不需要真实模型。它用一个离线假模型主动发起工具调用，从而完整演示 context 如何进入工具。真实 DeepSeek 工具调用可以参考项目里的 `langGraph/TestOne.py`，但 Notebook 里用离线模型避免接口 502 影响学习。

In [7]:
from __future__ import annotations

from collections.abc import Sequence
from dataclasses import dataclass
from typing import Any

from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.runnables import Runnable
from langchain_core.tools import BaseTool


@dataclass
class Context:
    user_id: str


USER_PROFILES = {
    "U001": "张三，上海销售",
    "U002": "李四，北京财务",
}


@tool
def get_user_profile(runtime: ToolRuntime[Context]) -> str:
    """读取当前登录用户资料。"""

    user_id = runtime.context.user_id
    profile = USER_PROFILES.get(user_id, "未知用户")
    return f"当前用户 {user_id}：{profile}。"


class ContextDemoModel(BaseChatModel):
    """离线演示用模型：第一次请求工具，第二次总结工具结果。"""

    calls: int = 0

    @property
    def _llm_type(self) -> str:
        return "context-demo-model"

    def bind_tools(
        self,
        tools: Sequence[dict[str, Any] | type | BaseTool | Any],
        *,
        tool_choice: str | None = None,
        **kwargs: Any,
    ) -> Runnable:
        return self

    def _generate(
        self,
        messages: list[BaseMessage],
        stop: list[str] | None = None,
        run_manager: Any | None = None,
        **kwargs: Any,
    ) -> ChatResult:
        self.calls += 1
        if self.calls == 1:
            message = AIMessage(
                content="",
                tool_calls=[
                    {
                        "name": "get_user_profile",
                        "args": {},
                        "id": "call-get-user-profile",
                        "type": "tool_call",
                    }
                ],
            )
        else:
            tool_message = next(
                (message for message in reversed(messages) if message.type == "tool"),
                None,
            )
            content = f"工具返回：{tool_message.content if tool_message else '无工具结果'}"
            message = AIMessage(content=content)
        return ChatResult(generations=[ChatGeneration(message=message)])


context_agent = create_agent(
    model=ContextDemoModel(),
    tools=[get_user_profile],
    context_schema=Context,
    system_prompt="你是一个演示 runtime context 的助手。",
)

response = context_agent.invoke(
    {"messages": [{"role": "user", "content": "读取当前用户资料"}]},
    context=Context(user_id="U001"),
)

print(response["messages"][-1].content)

工具返回：当前用户 U001：张三，上海销售。


上面的 context 传递链路是：

```text
Context(user_id="U001")
-> agent.invoke(..., context=context)
-> Agent 执行 get_user_profile 工具
-> ToolRuntime[Context]
-> runtime.context.user_id
-> "U001"
```

这就是 context 的核心价值：把运行时信息安全地交给工具，而不是让模型自己猜。

## 5. StateGraph：状态、节点、边

LangGraph 的底层核心是图：

- `State`：当前状态快照。
- `Node`：处理状态的 Python 函数。
- `Edge`：决定下一步去哪个节点。

下面是最小状态图。

In [ ]:
from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph


class SimpleState(TypedDict):
    question: str
    answer: str


def answer_node(state: SimpleState) -> dict[str, str]:
    return {"answer": f"你问的是：{state['question']}"}


builder = StateGraph(SimpleState)
builder.add_node("answer", answer_node)
builder.add_edge(START, "answer")
builder.add_edge("answer", END)

simple_graph = builder.compile()
simple_graph.invoke({"question": "什么是 LangGraph？"})

## 6. 条件边：根据状态决定下一步

固定边适合线性流程，条件边适合分支流程。下面的图会根据 `score` 判断走 `pass_node` 还是 `fail_node`。

In [ ]:
from typing import Literal
from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph


class ScoreState(TypedDict):
    score: int
    result: str


def route_by_score(state: ScoreState) -> Literal["pass_node", "fail_node"]:
    return "pass_node" if state["score"] >= 60 else "fail_node"


def pass_node(state: ScoreState) -> dict[str, str]:
    return {"result": "通过"}


def fail_node(state: ScoreState) -> dict[str, str]:
    return {"result": "未通过"}


builder = StateGraph(ScoreState)
builder.add_conditional_edges(START, route_by_score)
builder.add_node("pass_node", pass_node)
builder.add_node("fail_node", fail_node)
builder.add_edge("pass_node", END)
builder.add_edge("fail_node", END)

score_graph = builder.compile()
print(score_graph.invoke({"score": 88, "result": ""}))
print(score_graph.invoke({"score": 42, "result": ""}))

## 7. State reducer：让列表追加而不是覆盖

默认情况下，节点返回的新字段会覆盖旧值。对于消息列表、日志列表这类字段，通常希望追加。可以用 `Annotated[..., reducer]` 指定合并方式。

In [ ]:
from operator import add
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import END, START, StateGraph


class LogState(TypedDict):
    logs: Annotated[list[str], add]


def step_one(state: LogState) -> dict[str, list[str]]:
    return {"logs": ["执行 step_one"]}


def step_two(state: LogState) -> dict[str, list[str]]:
    return {"logs": ["执行 step_two"]}


builder = StateGraph(LogState)
builder.add_node("step_one", step_one)
builder.add_node("step_two", step_two)
builder.add_edge(START, "step_one")
builder.add_edge("step_one", "step_two")
builder.add_edge("step_two", END)

log_graph = builder.compile()
log_graph.invoke({"logs": ["开始"]})

## 8. 持久化：checkpointer + thread_id

短期记忆依赖检查点。核心是：

- 创建图时传入 `checkpointer`。
- 调用图时传入 `config={"configurable": {"thread_id": "..."}}`。
- 相同 `thread_id` 会接上同一条线程。

In [ ]:
from operator import add
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph


class MemoryState(TypedDict):
    events: Annotated[list[str], add]


def remember_node(state: MemoryState) -> dict[str, list[str]]:
    return {"events": ["本轮执行了一次"]}


builder = StateGraph(MemoryState)
builder.add_node("remember", remember_node)
builder.add_edge(START, "remember")
builder.add_edge("remember", END)

memory_graph = builder.compile(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "demo-thread"}}

first = memory_graph.invoke({"events": ["开始"]}, config=config)
second = memory_graph.invoke({"events": ["继续"]}, config=config)

print("第一次：", first)
print("第二次：", second)

## 9. 流式输出

LangGraph 可以边执行边输出中间结果。对于图来说，`.stream()` 会逐步返回节点更新。

In [ ]:
for chunk in simple_graph.stream({"question": "流式输出是什么？"}):
    print(chunk)

## 10. 人机协作、时间旅行、多智能体

这些能力都建立在“图状态 + 检查点”之上：

- 人机协作：在关键节点暂停，等待人工审批或补充信息。
- 时间旅行：从旧检查点恢复执行，观察不同路径。
- 多智能体：把不同 Agent 当作节点或子图，由主管或路由逻辑调度。

入门阶段建议先掌握：`create_agent`、`context_schema`、`StateGraph`、条件边、reducer、checkpointer。

## 11. 常见错误

| 问题 | 原因 | 处理 |
| --- | --- | --- |
| `create_react_agent` 弃用警告 | 还在从 `langgraph.prebuilt` 导入旧入口 | 改用 `from langchain.agents import create_agent` |
| 工具里拿不到 context | 直接调用 `tool.invoke`，没有走 Agent 工具节点 | 用 `agent.invoke(..., context=Context(...))` |
| Agent 不记得前文 | 没有 checkpointer 或 `thread_id` 变化 | 使用 `checkpointer + thread_id` |
| 节点返回值错误 | 节点没有返回状态更新 dict | 返回 `{"字段": 值}` |
| 列表被覆盖 | 没有 reducer | 使用 `Annotated[list[T], add]` 或消息 reducer |

## 12. 参考链接

- LangGraph 快速入门: <https://langgraph.com.cn/agents/agents.1.html>
- LangGraph 图 API 概念: <https://langgraph.com.cn/concepts/low_level.1.html>
- LangGraph 持久化: <https://langgraph.com.cn/concepts/persistence.1.html>
- LangGraph 记忆概念: <https://langgraph.com.cn/concepts/memory.1.html>
- LangGraph 人工参与循环: <https://langgraph.com.cn/concepts/human_in_the_loop.1.html>
- LangGraph 多代理系统: <https://langgraph.com.cn/concepts/multi_agent.1.html>